# Pipeline ETL — Foodiz

**Objectif** : Extraire, transformer et charger les donnees Open Food Facts dans MongoDB pour alimenter le systeme de recommandation par regime alimentaire.

**Etapes :**
1. **Extract** — Lecture rapide du CSV (colonnes utiles uniquement, moteur PyArrow)
2. **Transform** — Nettoyage, traitement des valeurs manquantes, normalisation, feature engineering (flags regime)
3. **Load** — Insertion des documents dans MongoDB

## 1. Imports et configuration

In [1]:
import pandas as pd
import numpy as np
from pymongo import MongoClient, InsertOne
from pymongo.errors import BulkWriteError
import time

import warnings
warnings.filterwarnings('ignore')

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "foodiz"
COLLECTION_NAME = "products"
CSV_PATH = "../Data/openfoodfacts-products.csv"
BATCH_SIZE = 10_000

---
# EXTRACT
---

## 2. Lecture rapide du CSV

In [2]:
cols_identite = ['code', 'product_name', 'brands', 'categories', 'categories_en', 'main_category_en']
cols_qualite = ['nutriscore_score', 'nutriscore_grade', 'nova_group']
cols_nutrition = [
    'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g',
    'carbohydrates_100g', 'sugars_100g', 'fiber_100g',
    'proteins_100g', 'salt_100g', 'sodium_100g'
]
cols_regime = [
    'ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags',
    'allergens', 'traces', 'traces_en',
    'labels', 'labels_tags', 'labels_en',
    'additives_n', 'additives_tags'
]
cols_meta = ['countries_en', 'image_url', 'completeness']

cols_utiles = cols_identite + cols_qualite + cols_nutrition + cols_regime + cols_meta

t0 = time.time()
df = pd.read_csv(
    CSV_PATH,
    sep="\t",
    usecols=cols_utiles,
    engine="pyarrow",
    on_bad_lines="skip"
)
duree = time.time() - t0
print(f"Extraction : {df.shape[0]:,} lignes x {df.shape[1]} colonnes en {duree:.1f}s")
print(f"Memoire : {df.memory_usage(deep=True).sum() / 1e9:.2f} Go")

Extraction : 4,535,040 lignes x 32 colonnes en 38.3s
Memoire : 3.04 Go


---
# TRANSFORM
---

## 3. Suppression des produits sans nom et sans donnees nutritionnelles

On filtre les produits inutilisables : pas de nom ou aucune valeur nutritionnelle renseignee.

In [3]:
n_before = len(df)

cols_nutrition_core = ['energy-kcal_100g', 'fat_100g', 'carbohydrates_100g', 'proteins_100g']

df = df[df['product_name'].notna() & (df['product_name'].str.strip() != '')]
df = df[df[cols_nutrition_core].notna().any(axis=1)]

df = df.drop_duplicates(subset='code', keep='first')

n_after = len(df)
print(f"Avant : {n_before:,} lignes")
print(f"Apres : {n_after:,} lignes")
print(f"Supprimees : {n_before - n_after:,} lignes ({(n_before - n_after) / n_before:.1%})")

Avant : 4,535,040 lignes
Apres : 2,200,323 lignes
Supprimees : 2,334,717 lignes (51.5%)


## 4. Nettoyage des valeurs aberrantes

Corriger les valeurs nutritionnelles impossibles (negatives ou depassant les limites physiques).

In [4]:
nutrient_bounds = {
    'energy-kcal_100g': (0, 900),
    'fat_100g': (0, 100),
    'saturated-fat_100g': (0, 100),
    'carbohydrates_100g': (0, 100),
    'sugars_100g': (0, 100),
    'fiber_100g': (0, 100),
    'proteins_100g': (0, 100),
    'salt_100g': (0, 100),
    'sodium_100g': (0, 40),
}

n_cleaned = 0
for col, (low, high) in nutrient_bounds.items():
    mask = df[col].notna() & ((df[col] < low) | (df[col] > high))
    n_cleaned += mask.sum()
    df.loc[mask, col] = np.nan

print(f"Valeurs aberrantes mises a NaN : {n_cleaned:,}")

Valeurs aberrantes mises a NaN : 7,883


## 5. Nettoyage du nutriscore et du groupe NOVA

In [5]:
valid_grades = {'a', 'b', 'c', 'd', 'e'}
df['nutriscore_grade'] = df['nutriscore_grade'].where(
    df['nutriscore_grade'].isin(valid_grades)
)

df['nova_group'] = pd.to_numeric(df['nova_group'], errors='coerce')
df['nova_group'] = df['nova_group'].where(df['nova_group'].isin([1, 2, 3, 4]))

print(f"Nutriscore grade valides : {df['nutriscore_grade'].notna().sum():,} ({df['nutriscore_grade'].notna().mean():.1%})")
print(f"NOVA group valides : {df['nova_group'].notna().sum():,} ({df['nova_group'].notna().mean():.1%})")

Nutriscore grade valides : 928,372 (42.2%)
NOVA group valides : 621,899 (28.3%)


## 6. Conversion des champs texte en listes (arrays)

Transformer les champs separes par des virgules en listes Python, qui deviendront des arrays MongoDB.

In [6]:
def split_and_clean(series, strip_prefix=False):
    """Transforme une colonne texte CSV en liste de valeurs nettoyees."""
    def _process(val):
        if pd.isna(val) or str(val).strip() == '':
            return []
        items = [item.strip() for item in str(val).split(',')]
        if strip_prefix:
            items = [item.split(':', 1)[-1] if ':' in item else item for item in items]
        items = [item.lower() for item in items if item]
        return list(dict.fromkeys(items))
    return series.apply(_process)

df['categories_list'] = split_and_clean(df['categories_en'])
df['labels_list'] = split_and_clean(df['labels_en'])
df['allergens_list'] = split_and_clean(df['allergens'], strip_prefix=True)
df['traces_list'] = split_and_clean(df['traces_en'])
df['additives_list'] = split_and_clean(df['additives_tags'], strip_prefix=True)
df['ingredients_analysis_list'] = split_and_clean(df['ingredients_analysis_tags'], strip_prefix=True)
df['countries_list'] = split_and_clean(df['countries_en'])

print("Exemples de conversion :")
sample = df[df['allergens_list'].apply(len) > 0].iloc[0]
print(f"  allergens brut   : {sample['allergens']}")
print(f"  allergens_list   : {sample['allergens_list']}")
sample2 = df[df['labels_list'].apply(len) > 0].iloc[0]
print(f"  labels brut      : {sample2['labels_en']}")
print(f"  labels_list      : {sample2['labels_list']}")

Exemples de conversion :
  allergens brut   : en:nuts
  allergens_list   : ['nuts']
  labels brut      : Organic,No GMOs,Non GMO project
  labels_list      : ['organic', 'no gmos', 'non gmo project']


## 7. Feature engineering — Flags de regime alimentaire

Creer des booleens pour faciliter le filtrage par regime dans le systeme de recommandation.

In [7]:
def has_tag(list_col, keywords):
    """Verifie si au moins un des mots-cles est present dans la liste."""
    return list_col.apply(lambda lst: any(kw in item for item in lst for kw in keywords))

def has_no_allergen(allergens_col, traces_col, allergen_keywords):
    """Verifie l'absence d'un allergene dans allergens ET traces."""
    def _check(row):
        all_items = row['allergens_list'] + row['traces_list']
        return not any(kw in item for item in all_items for kw in allergen_keywords)
    return df[['allergens_list', 'traces_list']].apply(_check, axis=1)

# Vegan : tag "vegan" dans l'analyse des ingredients OU dans les labels
df['is_vegan'] = (
    has_tag(df['ingredients_analysis_list'], ['vegan'])
    & ~has_tag(df['ingredients_analysis_list'], ['non-vegan'])
) | has_tag(df['labels_list'], ['vegan'])

# Vegetarien : vegan OU tag vegetarien
df['is_vegetarian'] = df['is_vegan'] | (
    has_tag(df['ingredients_analysis_list'], ['vegetarian'])
    & ~has_tag(df['ingredients_analysis_list'], ['non-vegetarian'])
) | has_tag(df['labels_list'], ['vegetarian'])

# Sans gluten : label "gluten-free" ou "no gluten" ET pas de gluten dans les allergenes
df['is_gluten_free'] = (
    has_tag(df['labels_list'], ['gluten-free', 'no gluten', 'sans gluten'])
    | has_no_allergen(df['allergens_list'], df['traces_list'], ['gluten', 'wheat', 'ble'])
)

# Sans lactose : label ou absence d'allergene lait
df['is_lactose_free'] = (
    has_tag(df['labels_list'], ['lactose-free', 'no lactose', 'sans lactose'])
    | has_no_allergen(df['allergens_list'], df['traces_list'], ['milk', 'lait', 'lactose'])
)

# Keto-friendly : riche en gras, pauvre en glucides
df['is_keto'] = (
    (df['fat_100g'] >= 40) &
    (df['carbohydrates_100g'] <= 10)
)

# Low-carb : glucides < 25g/100g
df['is_low_carb'] = df['carbohydrates_100g'] <= 25

# High-protein : proteines >= 20g/100g
df['is_high_protein'] = df['proteins_100g'] >= 20

# Low-sugar : sucres <= 5g/100g
df['is_low_sugar'] = df['sugars_100g'] <= 5

# Bio
df['is_organic'] = has_tag(df['labels_list'], ['organic', 'bio', 'biologique'])

flags = ['is_vegan', 'is_vegetarian', 'is_gluten_free', 'is_lactose_free',
         'is_keto', 'is_low_carb', 'is_high_protein', 'is_low_sugar', 'is_organic']

print("Nombre de produits par regime :")
for flag in flags:
    n = df[flag].sum()
    print(f"  {flag:25s} : {n:>10,} ({n / len(df):.1%})")

Nombre de produits par regime :
  is_vegan                  :    400,846 (18.2%)
  is_vegetarian             :    603,158 (27.4%)
  is_gluten_free            :  2,062,174 (93.7%)
  is_lactose_free           :  2,018,163 (91.7%)
  is_keto                   :     85,239 (3.9%)
  is_low_carb               :  1,306,104 (59.4%)
  is_high_protein           :    337,384 (15.3%)
  is_low_sugar              :  1,249,200 (56.8%)
  is_organic                :    184,453 (8.4%)


## 8. Construction du document MongoDB

Structurer chaque produit en document avec des sous-objets logiques (nutrition, regime, meta).

In [8]:
def row_to_document(row):
    """Convertit une ligne pandas en document MongoDB structure."""
    def clean_val(val):
        if isinstance(val, float) and np.isnan(val):
            return None
        return val

    doc = {
        'code': str(row.get('code', '')),
        'product_name': clean_val(row.get('product_name')),
        'brands': clean_val(row.get('brands')),
        'main_category': clean_val(row.get('main_category_en')),
        'categories': row.get('categories_list', []),
        'image_url': clean_val(row.get('image_url')),

        'nutrition': {},
        'quality': {},
        'regime': {},
        'meta': {}
    }

    nutrition_fields = {
        'energy_kcal': 'energy-kcal_100g',
        'fat': 'fat_100g',
        'saturated_fat': 'saturated-fat_100g',
        'carbohydrates': 'carbohydrates_100g',
        'sugars': 'sugars_100g',
        'fiber': 'fiber_100g',
        'proteins': 'proteins_100g',
        'salt': 'salt_100g',
        'sodium': 'sodium_100g',
    }
    for key, col in nutrition_fields.items():
        val = clean_val(row.get(col))
        if val is not None:
            doc['nutrition'][key] = round(val, 2)

    nutriscore = clean_val(row.get('nutriscore_grade'))
    if nutriscore:
        doc['quality']['nutriscore_grade'] = nutriscore
    score = clean_val(row.get('nutriscore_score'))
    if score is not None:
        doc['quality']['nutriscore_score'] = round(score, 1)
    nova = clean_val(row.get('nova_group'))
    if nova is not None:
        doc['quality']['nova_group'] = int(nova)

    doc['labels'] = row.get('labels_list', [])
    doc['allergens'] = row.get('allergens_list', [])
    doc['traces'] = row.get('traces_list', [])
    doc['additives'] = row.get('additives_list', [])
    doc['ingredients_text'] = clean_val(row.get('ingredients_text'))

    for flag in flags:
        val = row.get(flag)
        if val is True:
            doc['regime'][flag] = True
        else:
            doc['regime'][flag] = False

    doc['meta']['countries'] = row.get('countries_list', [])
    doc['meta']['completeness'] = round(clean_val(row.get('completeness')) or 0, 3)
    doc['meta']['additives_count'] = int(clean_val(row.get('additives_n')) or 0)

    doc = {k: v for k, v in doc.items() if v is not None and v != {} and v != []}

    return doc

sample_doc = row_to_document(df.iloc[0])
print("Exemple de document MongoDB :")
import json
print(json.dumps(sample_doc, indent=2, ensure_ascii=False, default=str))

Exemple de document MongoDB :
{
  "code": "1000.0",
  "product_name": "Pinto Bean",
  "brands": "Central Bean",
  "main_category": "Asian-style-ready-meal",
  "categories": [
    "asian-style-ready-meal"
  ],
  "image_url": "https://images.openfoodfacts.org/images/products/000/000/000/1000/front_en.48.400.jpg",
  "nutrition": {
    "energy_kcal": 261.4,
    "fat": 10.2,
    "saturated_fat": 3.6,
    "carbohydrates": 22.5,
    "sugars": 4.9,
    "fiber": 3.8,
    "proteins": 17.5
  },
  "regime": {
    "is_vegan": false,
    "is_vegetarian": false,
    "is_gluten_free": false,
    "is_lactose_free": false,
    "is_keto": false,
    "is_low_carb": false,
    "is_high_protein": false,
    "is_low_sugar": false,
    "is_organic": false
  },
  "meta": {
    "countries": [
      "united kingdom",
      "world"
    ],
    "completeness": 0.662,
    "additives_count": 0
  },
  "labels": [
    "organic",
    "no gmos",
    "non gmo project"
  ]
}


---
# LOAD
---

## 9. Connexion a MongoDB et creation de la collection

In [9]:
client = MongoClient(MONGO_URI)
db = client[DB_NAME]

if COLLECTION_NAME in db.list_collection_names():
    db[COLLECTION_NAME].drop()
    print(f"Collection '{COLLECTION_NAME}' existante supprimee.")

collection = db[COLLECTION_NAME]
print(f"Connecte a MongoDB : {MONGO_URI}")
print(f"Base : {DB_NAME} / Collection : {COLLECTION_NAME}")

Connecte a MongoDB : mongodb://localhost:27017
Base : foodiz / Collection : products


## 10. Insertion par batch dans MongoDB

In [10]:
t0 = time.time()
total = len(df)
inserted = 0
errors = 0

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    batch = df.iloc[start:end]

    docs = [row_to_document(row) for _, row in batch.iterrows()]
    operations = [InsertOne(doc) for doc in docs]

    try:
        result = collection.bulk_write(operations, ordered=False)
        inserted += result.inserted_count
    except BulkWriteError as e:
        inserted += e.details.get('nInserted', 0)
        errors += len(e.details.get('writeErrors', []))

    if (start // BATCH_SIZE) % 50 == 0:
        elapsed = time.time() - t0
        pct = end / total * 100
        speed = inserted / elapsed if elapsed > 0 else 0
        print(f"  [{pct:5.1f}%] {inserted:>10,} / {total:,} inseres ({speed:,.0f} docs/s)")

duree = time.time() - t0
print(f"\nChargement termine en {duree:.0f}s")
print(f"  Inseres : {inserted:,}")
print(f"  Erreurs : {errors:,}")
print(f"  Vitesse : {inserted / duree:,.0f} docs/s")
# Temps : 

  [  0.5%]     10,000 / 2,200,323 inseres (5,009 docs/s)
  [ 23.2%]    510,000 / 2,200,323 inseres (5,550 docs/s)
  [ 45.9%]  1,010,000 / 2,200,323 inseres (5,332 docs/s)
  [ 68.6%]  1,510,000 / 2,200,323 inseres (5,169 docs/s)
  [ 91.4%]  2,010,000 / 2,200,323 inseres (5,110 docs/s)

Chargement termine en 432s
  Inseres : 2,200,323
  Erreurs : 0
  Vitesse : 5,097 docs/s


## 11. Creation des index pour les performances de requetes

In [11]:
indexes = [
    ('code', {'unique': True}),
    ('quality.nutriscore_grade', {}),
    ('quality.nova_group', {}),
    ('categories', {}),
    ('allergens', {}),
    ('labels', {}),
    ('regime.is_vegan', {}),
    ('regime.is_vegetarian', {}),
    ('regime.is_gluten_free', {}),
    ('regime.is_keto', {}),
    ('regime.is_low_carb', {}),
    ('regime.is_high_protein', {}),
    ('nutrition.energy_kcal', {}),
    ('nutrition.proteins', {}),
    ('nutrition.carbohydrates', {}),
    ('nutrition.fat', {}),
    ('product_name', {'name': 'idx_product_name'}),
]

print("Creation des index :")
for field, kwargs in indexes:
    idx_name = collection.create_index(field, **kwargs)
    print(f"  {idx_name}")

print(f"\nTotal index : {len(collection.index_information())}")
# Temps : 

Creation des index :
  code_1
  quality.nutriscore_grade_1
  quality.nova_group_1
  categories_1
  allergens_1
  labels_1
  regime.is_vegan_1
  regime.is_vegetarian_1
  regime.is_gluten_free_1
  regime.is_keto_1
  regime.is_low_carb_1
  regime.is_high_protein_1
  nutrition.energy_kcal_1
  nutrition.proteins_1
  nutrition.carbohydrates_1
  nutrition.fat_1
  idx_product_name

Total index : 18


## 12. Verification du chargement

In [12]:
print(f"Documents dans la collection : {collection.count_documents({}):,}")
print(f"Taille de la collection : {db.command('collStats', COLLECTION_NAME)['size'] / 1e9:.2f} Go")
print()

print("Exemple de document :")
sample = collection.find_one({'nutrition': {'$exists': True}, 'quality.nutriscore_grade': {'$exists': True}})
print(json.dumps(sample, indent=2, ensure_ascii=False, default=str))
# Temps : 

Documents dans la collection : 2,200,323
Taille de la collection : 1.86 Go

Exemple de document :
{
  "_id": "6a42704e3ff6d35f7878b27f",
  "code": "101266509.0",
  "product_name": "Familjepack Schnitzel Frysta",
  "brands": "Hälsans Kök",
  "main_category": "Vegan patties",
  "categories": [
    "plant-based foods and beverages",
    "plant-based foods",
    "meat alternatives",
    "meat analogues",
    "vegetarian patties",
    "vegan patties"
  ],
  "image_url": "https://images.openfoodfacts.org/images/products/000/010/126/6509/front_en.8.400.jpg",
  "nutrition": {
    "energy_kcal": 223.0,
    "fat": 12.0,
    "saturated_fat": 1.2,
    "carbohydrates": 11.8,
    "sugars": 0.6,
    "fiber": 6.2,
    "proteins": 13.5,
    "salt": 0.8,
    "sodium": 0.32
  },
  "quality": {
    "nutriscore_grade": "a",
    "nutriscore_score": -2.0,
    "nova_group": 4
  },
  "regime": {
    "is_vegan": true,
    "is_vegetarian": true,
    "is_gluten_free": true,
    "is_lactose_free": true,
    "is_ke

## 13. Tests de requetes de recommandation

Verifier que les index fonctionnent et que les requetes par regime sont rapides.

In [13]:
queries = {
    "Produits vegan": {
        'regime.is_vegan': True
    },
    "Produits keto (nutriscore A ou B)": {
        'regime.is_keto': True,
        'quality.nutriscore_grade': {'$in': ['a', 'b']}
    },
    "Produits sans gluten, high protein": {
        'regime.is_gluten_free': True,
        'regime.is_high_protein': True
    },
    "Produits bio, low sugar, < 200 kcal": {
        'regime.is_organic': True,
        'regime.is_low_sugar': True,
        'nutrition.energy_kcal': {'$lte': 200}
    },
    "Produits sans allergene lait ni gluten": {
        'allergens': {'$nin': ['milk', 'gluten']},
        'traces': {'$nin': ['milk', 'gluten']}
    }
}

print("Tests de requetes :")
print("=" * 70)
for name, query in queries.items():
    t0 = time.time()
    count = collection.count_documents(query)
    duree = (time.time() - t0) * 1000
    print(f"\n{name}")
    print(f"  Requete  : {query}")
    print(f"  Resultats: {count:,} produits")
    print(f"  Temps    : {duree:.0f} ms")
    
# Temps : 

Tests de requetes :

Produits vegan
  Requete  : {'regime.is_vegan': True}
  Resultats: 400,846 produits
  Temps    : 148 ms

Produits keto (nutriscore A ou B)
  Requete  : {'regime.is_keto': True, 'quality.nutriscore_grade': {'$in': ['a', 'b']}}
  Resultats: 16,722 produits
  Temps    : 254 ms

Produits sans gluten, high protein
  Requete  : {'regime.is_gluten_free': True, 'regime.is_high_protein': True}
  Resultats: 331,206 produits
  Temps    : 892 ms

Produits bio, low sugar, < 200 kcal
  Requete  : {'regime.is_organic': True, 'regime.is_low_sugar': True, 'nutrition.energy_kcal': {'$lte': 200}}
  Resultats: 46,655 produits
  Temps    : 4070 ms

Produits sans allergene lait ni gluten
  Requete  : {'allergens': {'$nin': ['milk', 'gluten']}, 'traces': {'$nin': ['milk', 'gluten']}}
  Resultats: 1,965,224 produits
  Temps    : 9108 ms


In [14]:
print("Exemple : 5 produits keto avec le meilleur nutriscore\n")
results = collection.find(
    {'regime.is_keto': True, 'quality.nutriscore_grade': {'$exists': True}},
    {'_id': 0, 'product_name': 1, 'brands': 1, 'quality.nutriscore_grade': 1,
     'nutrition.fat': 1, 'nutrition.carbohydrates': 1, 'nutrition.proteins': 1}
).sort('quality.nutriscore_grade', 1).limit(5)

for doc in results:
    n = doc.get('nutrition', {})
    q = doc.get('quality', {})
    print(f"  [{q.get('nutriscore_grade', '?').upper()}] {doc.get('product_name', 'N/A')} "
          f"({doc.get('brands', 'N/A')}) — "
          f"F:{n.get('fat', '?')}g C:{n.get('carbohydrates', '?')}g P:{n.get('proteins', '?')}g")
    
# Temps : 

Exemple : 5 produits keto avec le meilleur nutriscore

  [A] pumpkin seeds (SebaGarden) — F:45.8g C:2.3g P:35.0g
  [A] Pistaches en poudre (Alsa) — F:56.0g C:8.1g P:18.0g
  [A] Graines de citrouille écalées crues (Mayrand chef) — F:49.0g C:10.0g P:30.0g
  [A] Nut mix (Trader Joe's) — F:53.0g C:8.4g P:20.0g
  [A] Almonds - Natural (Wonderful) — F:50.0g C:9.1g P:21.0g


## 14. Resume du pipeline ETL

| Etape | Description | Detail |
|-------|------------|--------|
| **Extract** | Lecture du CSV | PyArrow + usecols (32 colonnes / 211) |
| **Transform — Filtrage** | Suppression des lignes inutilisables | Pas de nom, pas de nutrition, doublons |
| **Transform — Nettoyage** | Valeurs aberrantes | Bornes physiques par nutriment |
| **Transform — Normalisation** | Nutriscore / NOVA | Filtrage des valeurs invalides |
| **Transform — Arrays** | Champs multi-valeurs | Conversion CSV → listes (allergens, labels, etc.) |
| **Transform — Features** | Flags de regime | 9 booleens (vegan, keto, gluten-free, etc.) |
| **Load — Insertion** | MongoDB | Bulk insert par batch de 10K documents |
| **Load — Index** | Optimisation | 17 index sur les champs de requete |